# Trout Age4 SimCLR Comparison

This notebook compares the existing image-only results against a SimCLR-pretrained ResNet18 backbone for the collapsed trout age task.

Target labels:
- `0+`
- `1+`
- `2+`
- `3+` = original ages `3`, `4`, and `5`

Input information:
- scale images only;
- no fish length;
- no fish weight.

The split is fish-level using `fish_key`, matching the other age4 image-only notebooks.

## 1. Setup

Run these first:

1. `trout_new_dataset_eda.ipynb`
2. `trout_texture_feature_extraction.ipynb` with `RUN_SAMPLE=False`
3. optionally `trout_age4_image_only_models.ipynb` to generate the ImageNet CNN comparison table

This notebook can still run without step 3; it will just report SimCLR results alone.

In [ ]:
from __future__ import annotations

import os
import random
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 140)

ROOT_DIR = Path(os.environ.get("TROUT_ROOT_DIR", "/home/jlc3q/data/Trout"))
CODE_DIR = Path(os.environ.get("TROUT_CODE_DIR", str(ROOT_DIR / "code_new")))
FEATURE_OUTPUT_DIR = CODE_DIR / "feature_outputs"
MODEL_OUTPUT_DIR = CODE_DIR / "model_outputs"
CHECKPOINT_DIR = MODEL_OUTPUT_DIR / "checkpoints"
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

MASTER_TEXTURE_CSV = FEATURE_OUTPUT_DIR / "master_with_texture_features_full.csv"
IMAGE_ONLY_RESULTS_CSV = MODEL_OUTPUT_DIR / "age4_image_only_model_results.csv"

SEED = 100
TEST_SIZE = 0.2
CLASS_NAMES = ["0+", "1+", "2+", "3+"]

SIMCLR_EPOCHS = 10
CLASSIFIER_EPOCHS = 10
SIMCLR_BATCH_SIZE = 128
CLASSIFIER_BATCH_SIZE = 64
SIMCLR_LR = 3e-4
CLASSIFIER_LR = 1e-4
TEMPERATURE = 0.2

# Keep this False for the fairest direct comparison: no test fish are used during self-supervised pretraining.
# Set True only if you explicitly want to use unlabeled/readable image pools for pretraining.
USE_EXTRA_UNLABELED_FOR_SIMCLR = False

random.seed(SEED)
np.random.seed(SEED)

print("ROOT_DIR:", ROOT_DIR)
print("CODE_DIR:", CODE_DIR)
print("MASTER_TEXTURE_CSV exists:", MASTER_TEXTURE_CSV.exists())
print("IMAGE_ONLY_RESULTS_CSV exists:", IMAGE_ONLY_RESULTS_CSV.exists())

## 2. Load Age4 Image Rows

The table comes from the texture extraction notebook, but SimCLR itself only uses the image path and age4 label.

In [ ]:
if not MASTER_TEXTURE_CSV.exists():
    raise FileNotFoundError(
        f"Missing {MASTER_TEXTURE_CSV}. Run trout_texture_feature_extraction.ipynb with RUN_SAMPLE=False first."
    )

master_df = pd.read_csv(MASTER_TEXTURE_CSV)
required_cols = {"path", "scale_id", "fish_key", "age4"}
missing_cols = required_cols - set(master_df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {sorted(missing_cols)}")

model_df = master_df[master_df["age4"].notna()].copy()
model_df["age4"] = model_df["age4"].astype(int)
model_df["path_exists"] = model_df["path"].map(lambda p: Path(str(p)).exists())
model_df = model_df[model_df["path_exists"]].copy().reset_index(drop=True)

print("model_df:", model_df.shape)
print("unique fish:", model_df["fish_key"].nunique())
print("age4 counts:")
display(model_df["age4"].value_counts().sort_index())
display(model_df[["scale_id", "fish_key", "age4", "path"]].head())

## 3. Fish-Level Split

This reuses the same deterministic split policy as the image-only comparison notebook.

In [ ]:
splitter = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=SEED)
train_idx, test_idx = next(splitter.split(model_df, model_df["age4"], groups=model_df["fish_key"]))
train_df = model_df.iloc[train_idx].copy().reset_index(drop=True)
test_df = model_df.iloc[test_idx].copy().reset_index(drop=True)

fish_overlap = sorted(set(train_df["fish_key"]) & set(test_df["fish_key"]))
print("train:", train_df.shape, train_df["age4"].value_counts().sort_index().to_dict())
print("test :", test_df.shape, test_df["age4"].value_counts().sort_index().to_dict())
print("Fish overlap:", len(fish_overlap))
assert len(fish_overlap) == 0

## 4. Torch Setup

In [ ]:
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    import torchvision.transforms as T
    import torchvision.models as tv_models
    from torch.utils.data import Dataset, DataLoader
    from PIL import Image
    from tqdm.auto import tqdm
except ImportError as exc:
    raise ImportError(
        "This notebook needs torch, torchvision, pillow, and tqdm in the active Jupyter kernel."
    ) from exc

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_CUDA = device.type == "cuda"
NUM_WORKERS = 0
PIN_MEMORY = USE_CUDA


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()
print("device:", device)
print("num_workers:", NUM_WORKERS)

## 5. Datasets, Transforms, and SimCLR Loss

In [ ]:
simclr_transform = T.Compose([
    T.Resize((224, 224)),
    T.RandomResizedCrop(224, scale=(0.65, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.RandomRotation(20),
    T.ColorJitter(brightness=0.25, contrast=0.25),
    T.RandomGrayscale(p=0.2),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class SimCLRDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]["path"]).convert("RGB")
        return self.transform(img), self.transform(img)

class ImageLabelDataset(Dataset):
    def __init__(self, dataframe: pd.DataFrame, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.y = self.df["age4"].astype(int).to_numpy()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img = Image.open(self.df.iloc[idx]["path"]).convert("RGB")
        return self.transform(img), int(self.y[idx])


def make_resnet18_backbone(imagenet_init: bool = True):
    try:
        weights = tv_models.ResNet18_Weights.DEFAULT if imagenet_init else None
        model = tv_models.resnet18(weights=weights)
    except AttributeError:
        model = tv_models.resnet18(pretrained=imagenet_init)
    model.fc = nn.Identity()
    return model

class NTXentLoss(nn.Module):
    def __init__(self, temperature: float = TEMPERATURE):
        super().__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss(reduction="sum")

    def forward(self, z1, z2):
        batch_size = z1.size(0)
        z = torch.cat((z1, z2), dim=0)
        sim = F.cosine_similarity(z.unsqueeze(1), z.unsqueeze(0), dim=2)

        positive = torch.cat([
            torch.diag(sim, batch_size),
            torch.diag(sim, -batch_size),
        ], dim=0).unsqueeze(1)

        mask = torch.ones((2 * batch_size, 2 * batch_size), dtype=torch.bool, device=z.device)
        mask.fill_diagonal_(False)
        for i in range(batch_size):
            mask[i, batch_size + i] = False
            mask[batch_size + i, i] = False

        negative = sim[mask].view(2 * batch_size, -1)
        logits = torch.cat([positive, negative], dim=1) / self.temperature
        labels = torch.zeros(2 * batch_size, dtype=torch.long, device=z.device)
        return self.criterion(logits, labels) / (2 * batch_size)

print("SimCLR components ready")

## 6. SimCLR Pretraining

By default, pretraining uses only the training fish split. This is the cleanest comparison against supervised ImageNet ResNet18 results.

In [ ]:
if USE_EXTRA_UNLABELED_FOR_SIMCLR:
    train_fish = set(train_df["fish_key"])
    simclr_df = master_df[master_df["fish_key"].isin(train_fish)].copy()
    simclr_df = simclr_df[simclr_df["path"].map(lambda p: Path(str(p)).exists())].reset_index(drop=True)
else:
    simclr_df = train_df.copy()

print("simclr_df:", simclr_df.shape)
print("unique fish for SimCLR:", simclr_df["fish_key"].nunique())
print("uses extra unlabeled images:", USE_EXTRA_UNLABELED_FOR_SIMCLR)

In [ ]:
def train_simclr_backbone(dataframe: pd.DataFrame, epochs: int = SIMCLR_EPOCHS):
    set_seed()
    dataset = SimCLRDataset(dataframe, simclr_transform)
    loader = DataLoader(
        dataset,
        batch_size=SIMCLR_BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        drop_last=False,
    )

    backbone = make_resnet18_backbone(imagenet_init=True).to(device)
    projection_head = nn.Sequential(
        nn.Linear(512, 128),
        nn.ReLU(),
        nn.Linear(128, 128),
    ).to(device)

    optimizer = torch.optim.AdamW(
        list(backbone.parameters()) + list(projection_head.parameters()),
        lr=SIMCLR_LR,
        weight_decay=1e-4,
    )
    criterion = NTXentLoss(temperature=TEMPERATURE)

    backbone.train()
    projection_head.train()
    for epoch in range(epochs):
        losses = []
        for x1, x2 in tqdm(loader, desc=f"SimCLR {epoch + 1}/{epochs}"):
            x1 = x1.to(device)
            x2 = x2.to(device)
            z1 = projection_head(backbone(x1))
            z2 = projection_head(backbone(x2))
            loss = criterion(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")

    ckpt_path = CHECKPOINT_DIR / "age4_simclr_resnet18_backbone.pt"
    torch.save({
        "backbone_state_dict": backbone.state_dict(),
        "simclr_epochs": epochs,
        "simclr_batch_size": SIMCLR_BATCH_SIZE,
        "temperature": TEMPERATURE,
        "use_extra_unlabeled": USE_EXTRA_UNLABELED_FOR_SIMCLR,
    }, ckpt_path)
    print("saved:", ckpt_path)
    return backbone

simclr_backbone = train_simclr_backbone(simclr_df, epochs=SIMCLR_EPOCHS)

## 7. Supervised Age4 Classifier on SimCLR Backbone

In [ ]:
class Age4Classifier(nn.Module):
    def __init__(self, backbone: nn.Module, n_classes: int = 4):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_classes),
        )

    def forward(self, image):
        return self.head(self.backbone(image))


def class_weight_tensor(y: pd.Series) -> torch.Tensor:
    counts = y.value_counts().reindex([0, 1, 2, 3], fill_value=0).astype(float)
    weights = counts.sum() / (len(counts) * counts.clip(lower=1))
    return torch.tensor(weights.to_numpy(), dtype=torch.float32, device=device)


def train_age4_classifier(backbone: nn.Module, epochs: int = CLASSIFIER_EPOCHS):
    set_seed()
    train_dataset = ImageLabelDataset(train_df, eval_transform)
    test_dataset = ImageLabelDataset(test_df, eval_transform)
    train_loader = DataLoader(train_dataset, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
    test_loader = DataLoader(test_dataset, batch_size=CLASSIFIER_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

    model = Age4Classifier(backbone).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(train_df["age4"]))
    optimizer = torch.optim.AdamW(model.parameters(), lr=CLASSIFIER_LR, weight_decay=1e-4)

    for epoch in range(epochs):
        model.train()
        losses = []
        for images, y in tqdm(train_loader, desc=f"SimCLR classifier {epoch + 1}/{epochs}"):
            images = images.to(device)
            y = y.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), y)
            loss.backward()
            optimizer.step()
            losses.append(float(loss.detach().cpu()))
        print(f"epoch={epoch + 1} loss={np.mean(losses):.4f}")

    ckpt_path = CHECKPOINT_DIR / "age4_simclr_resnet18_classifier.pt"
    torch.save({
        "model_state_dict": model.state_dict(),
        "classifier_epochs": epochs,
        "class_names": CLASS_NAMES,
    }, ckpt_path)
    print("saved:", ckpt_path)
    return model, test_loader

simclr_model, simclr_test_loader = train_age4_classifier(simclr_backbone, epochs=CLASSIFIER_EPOCHS)

## 8. Evaluation

In [ ]:
def evaluate_predictions(name: str, y_true, y_pred) -> dict:
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    report_dict = classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
        "support": int(len(y_true)),
    }

    print("\n===", name, "===")
    print("Accuracy:", round(metrics["accuracy"], 4))
    print("Balanced accuracy:", round(metrics["balanced_accuracy"], 4))
    print("Macro F1:", round(metrics["macro_f1"], 4))
    print(classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2, 3],
        target_names=CLASS_NAMES,
        digits=4,
        zero_division=0,
    ))
    return metrics

@torch.no_grad()
def evaluate_torch_model(model: nn.Module, loader: DataLoader, name: str):
    model.eval()
    y_true = []
    y_pred = []
    for images, y in tqdm(loader, desc=f"evaluate {name}"):
        logits = model(images.to(device))
        pred = logits.argmax(dim=1).detach().cpu().numpy()
        y_pred.extend(pred.tolist())
        y_true.extend(y.numpy().tolist())
    metrics = evaluate_predictions(name, y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])
    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_path = MODEL_OUTPUT_DIR / f"confusion_matrix_{name}.csv"
    cm_df.to_csv(cm_path)
    print("saved:", cm_path)
    display(cm_df)
    return metrics, np.asarray(y_pred), cm_df

simclr_metrics, simclr_pred, simclr_cm = evaluate_torch_model(simclr_model, simclr_test_loader, "simclr_resnet18_age4")

## 9. Compare Against Existing Image-Only Results

In [ ]:
simclr_results_df = pd.DataFrame([simclr_metrics])
simclr_results_path = MODEL_OUTPUT_DIR / "age4_simclr_results.csv"
simclr_results_df.to_csv(simclr_results_path, index=False)
print("saved:", simclr_results_path)

if IMAGE_ONLY_RESULTS_CSV.exists():
    previous_results = pd.read_csv(IMAGE_ONLY_RESULTS_CSV)
    comparison_df = pd.concat([previous_results, simclr_results_df], ignore_index=True)
else:
    print("No previous image-only results found; showing SimCLR only.")
    comparison_df = simclr_results_df.copy()

comparison_df = comparison_df.sort_values("balanced_accuracy", ascending=False)
comparison_path = MODEL_OUTPUT_DIR / "age4_image_only_plus_simclr_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
print("saved:", comparison_path)
display(comparison_df)

## 10. Run Summary

In [ ]:
summary = {
    "n_rows": int(len(model_df)),
    "n_train_rows": int(len(train_df)),
    "n_test_rows": int(len(test_df)),
    "n_unique_fish": int(model_df["fish_key"].nunique()),
    "n_train_fish": int(train_df["fish_key"].nunique()),
    "n_test_fish": int(test_df["fish_key"].nunique()),
    "fish_overlap": int(len(set(train_df["fish_key"]) & set(test_df["fish_key"]))),
    "simclr_epochs": SIMCLR_EPOCHS,
    "classifier_epochs": CLASSIFIER_EPOCHS,
    "simclr_batch_size": SIMCLR_BATCH_SIZE,
    "classifier_batch_size": CLASSIFIER_BATCH_SIZE,
    "temperature": TEMPERATURE,
    "use_extra_unlabeled_for_simclr": USE_EXTRA_UNLABELED_FOR_SIMCLR,
    "uses_length_weight": False,
}
summary_path = MODEL_OUTPUT_DIR / "age4_simclr_run_summary.json"
summary_path.write_text(__import__("json").dumps(summary, indent=2))
print("saved:", summary_path)
summary